In [15]:
import speech_recognition as sr

In [16]:
import pandas as pd
df=pd.read_csv("D:\ds projects\question_set.csv",encoding='ISO-8859-1')
df

,sr_no,question,ans,ans1
0,1.0,What is Python? List some popular applications...,"Python is a widely-used general-purpose, high-...","Python is a widely-used general-purpose, high-..."
1,2.0,What are the benefits of using Python language...,Object-Oriented Language\nHigh-Level Language\...,Object-Oriented Language\nHigh-Level Language\...
2,3.0,What is the difference between a Mutable datat...,"Mutable data types can be edited i.e., they ca...","Mutable data types can be edited i.e., they ca..."
3,4.0,What is the difference between a Set and Dicti...,The set is an unordered collection of data typ...,The set is an unordered collection of data typ...
4,5.0,What is a pass in Python?,Pass means performing no operation or in other...,Pass means performing no operation or in other...
5,6.0,Difference between for loop and while loop in ...,The for Loop is generally used to iterate th...,The for Loop is generally used to iterate th...
6,NaN,NaN,NaN,NaN


In [17]:
df.isnull().sum()

sr_no       1
question    1
ans         1
ans1        1
dtype: int64

In [18]:
df.fillna("Not Mentioned",)

,sr_no,question,ans,ans1
0,1.0,What is Python? List some popular applications...,"Python is a widely-used general-purpose, high-...","Python is a widely-used general-purpose, high-..."
1,2.0,What are the benefits of using Python language...,Object-Oriented Language\nHigh-Level Language\...,Object-Oriented Language\nHigh-Level Language\...
2,3.0,What is the difference between a Mutable datat...,"Mutable data types can be edited i.e., they ca...","Mutable data types can be edited i.e., they ca..."
3,4.0,What is the difference between a Set and Dicti...,The set is an unordered collection of data typ...,The set is an unordered collection of data typ...
4,5.0,What is a pass in Python?,Pass means performing no operation or in other...,Pass means performing no operation or in other...
5,6.0,Difference between for loop and while loop in ...,The for Loop is generally used to iterate th...,The for Loop is generally used to iterate th...
6,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned


In [19]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [20]:
def preprocess_text(text):
    text=text.lower()
    words=word_tokenize(text)
    stop_word=set(stopwords.words('english'))
    words=[i for i in words if i not in stop_word]
    stemmer=PorterStemmer()
    words=[stemmer.stem(i) for i in words]
    preprocessed_text=' '.join(words)
    return preprocessed_text


In [21]:
s=" I am Smit working as Data Scientist"
preprocess_text(s)

'smit work data scientist'

In [22]:
df['ans1'] = df['ans1'].apply(preprocess_text)
df.head(1)

AttributeError: 'float' object has no attribute 'lower'

In [9]:
import pickle
with open('preprocess_text.pkl','wb')as file:
    pickle.dump(df,file)


In [10]:
df['question']=df['question'].str.strip()

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
model=SentenceTransformer('paraphrase-MiniLM-L6-v2')

In [25]:
def suggest_sections(ans,df,min_suggestion=1):
    preprocessed_ans=preprocess_text(ans)
    ans_emedding=model.emcode(preprocessed_ans)
    section_embedding=model.encode(df['ans1'].tolist())
    similarities=util.pytorch_cos_sim(ans_embedding,section_embedding)
    similarity_threhold=0.2
    relevant_indices=[]
    while len(relevant_indices<min_suggestion and similarity_threhold>0):
        relevant_indices=[ i for i,sim in enumerate(similarities) if sim>similarity_threhold]
        similarity_threhold-=0.5
        sorted_indices=sorted(relevant_indices,key=lambda i: similarities[i],reverse=True )
        suggestions=[{'index':i,
                      'question':df.iloc[i]['question'],'ans':df.iloc[i]['ans'],'similarity_score':similarities[0][1].item()}for i in sorted_indices]
        
        return suggestions
        



